In [1]:
from pathlib import Path
import json
import re
from collections import Counter
import math
from typing import List, Dict, Any, Optional


# Tokenisation simple (robuste pour FR + chiffres)
def tokenize(text: str) -> List[str]:
    text = (text or "").lower()
    return re.findall(r"[a-zàâçéèêëîïôùûüÿñæœ0-9]+", text)

# 1) Build index BM25 global
def build_bm25_global_index(
    chunks_dir: Path = Path("../data/chunks"),
    out_path: Path = Path("../data/bm25/bm25_global.json"),
    k1: float = 1.5,
    b: float = 0.75,
) -> Dict[str, Any]:
    """
    Construit un index BM25 GLOBAL sur tous les chunks présents dans ../data/chunks/*.json
    et sauvegarde un fichier unique: ../data/bm25/bm25_global.json

    Le fichier contient :
    - paramètres BM25
    - stats corpus (N, avgdl)
    - chunks (doc_name, chunk_id, content, tokens_len)
    - idf global
    - tf par chunk (Counter -> dict)
    - doc_len par chunk

    Retourne le dict d'index (JSON-serializable).
    """
    if not chunks_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {chunks_dir}")

    chunk_files = sorted(chunks_dir.glob("*.json"))
    if not chunk_files:
        raise FileNotFoundError(f"Aucun fichier .json dans : {chunks_dir}")

    print(f"🚀 Build BM25 GLOBAL sur {len(chunk_files)} fichiers chunks")

    all_chunks: List[Dict[str, Any]] = []
    tokenized_chunks: List[List[str]] = []
    doc_lens: List[int] = []
    df = Counter()  # document frequency = nb de chunks contenant le terme

    # 1) Charger + tokeniser tous les chunks
    total_chunks = 0
    for i, path in enumerate(chunk_files, start=1):
        doc_name = path.stem
        data = json.loads(path.read_text(encoding="utf-8"))

        if not isinstance(data, list):
            raise ValueError(f"Fichier chunks invalide (pas une liste) : {path}")

        print(f"📄 [{i}/{len(chunk_files)}] {doc_name} → {len(data)} chunks")
        for j, c in enumerate(data):
            content = c.get("content", "") or ""
            chunk_id = c.get("chunk_id", j)

            toks = tokenize(content)
            tokenized_chunks.append(toks)
            doc_lens.append(len(toks))

            # df: compte 1 fois par chunk
            for t in set(toks):
                df[t] += 1

            all_chunks.append({
                "doc_name": doc_name,
                "chunk_id": chunk_id,
                "content": content,
                "tokens_len": len(toks),
            })
            total_chunks += 1

    if total_chunks == 0:
        raise ValueError("Aucun chunk chargé (corpus vide)")

    N = total_chunks
    avgdl = sum(doc_lens) / max(1, N)

    # 2) Calcul IDF global
    idf = {}
    for t, dft in df.items():
        # IDF BM25 standard (avec +1.0 pour éviter valeurs négatives)
        idf[t] = math.log((N - dft + 0.5) / (dft + 0.5) + 1.0)

    # 3) Calcul TF par chunk
    tf = [dict(Counter(toks)) for toks in tokenized_chunks]

    index = {
        "type": "bm25_global",
        "params": {"k1": k1, "b": b},
        "stats": {
            "N_chunks": N,
            "avgdl": avgdl,
            "N_terms": len(idf),
            "source_dir": str(chunks_dir),
        },
        "chunks": all_chunks,
        "idf": idf,
        "tf": tf,
        "doc_len": doc_lens,
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"\n✅ Index BM25 global écrit → {out_path}")
    print(f"   - chunks: {N}")
    print(f"   - termes: {len(idf)}")
    print(f"   - avgdl : {avgdl:.2f}")


In [2]:
build_bm25_global_index()

🚀 Build BM25 GLOBAL sur 7 fichiers chunks
📄 [1/7] GDO-Interventions-en-milieu-agricole-2019-V2 → 55 chunks
📄 [2/7] GDO-interventions-silos-VF-09-2019 → 41 chunks
📄 [3/7] GDO-Operations-Presence-Electricite → 92 chunks
📄 [4/7] GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC → 34 chunks
📄 [5/7] GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 → 77 chunks
📄 [6/7] GDO_interventions_a_bord_bateaux_en_eaux_interieures → 46 chunks
📄 [7/7] GDO_Interventions_dans_les_eoliennes_2019 → 18 chunks

✅ Index BM25 global écrit → ..\data\bm25\bm25_global.json
   - chunks: 363
   - termes: 10911
   - avgdl : 371.44


In [3]:

def load_bm25_global_index(
    path: Path = Path("../data/bm25/bm25_global.json")
) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Index BM25 introuvable : {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def bm25_score_one_chunk(
    q_tokens: List[str],
    tf_chunk: Dict[str, int],
    doc_len: int,
    idf: Dict[str, float],
    avgdl: float,
    k1: float,
    b: float,
) -> float:
    """
    Score BM25 d'un chunk pour une requête tokenisée.
    """
    score = 0.0
    denom_const = k1 * (1.0 - b + b * (doc_len / (avgdl if avgdl > 0 else 1.0)))

    for t in q_tokens:
        if t not in tf_chunk:
            continue
        tf = tf_chunk[t]
        # terme absent de l'idf (rare) => 0
        idf_t = idf.get(t, 0.0)
        score += idf_t * (tf * (k1 + 1.0)) / (tf + denom_const)

    return float(score)

def search_top_chunks_bm25(
    question: str,
    top_k: int = 5,
    index: Optional[Dict[str, Any]] = None,
    index_path: Path = Path("../data/bm25/bm25_global.json"),
) -> List[Dict[str, Any]]:
    """
    Retourne les top_k chunks selon BM25 global.
    Format :
    [{rank, score, doc_name, chunk_id, content}]
    """
    if index is None:
        index = load_bm25_global_index(index_path)

    q_tokens = tokenize(question)
    k1 = index["params"]["k1"]
    b = index["params"]["b"]
    avgdl = index["stats"]["avgdl"]
    idf = index["idf"]

    scores = []
    for i in range(index["stats"]["N_chunks"]):
        s = bm25_score_one_chunk(
            q_tokens=q_tokens,
            tf_chunk=index["tf"][i],
            doc_len=index["doc_len"][i],
            idf=idf,
            avgdl=avgdl,
            k1=k1,
            b=b,
        )
        scores.append((s, i))

    # top-k
    scores.sort(key=lambda x: x[0], reverse=True)
    top = scores[:top_k]

    results = []
    for rank, (s, i) in enumerate(top, start=1):
        c = index["chunks"][i]
        results.append({
            "rank": rank,
            "score": s,
            "doc_name": c["doc_name"],
            "chunk_id": c["chunk_id"],
            "content": c["content"],
        })
    return results

In [4]:
question = "Que faire en cas de cable électrique tombé au sol ?"
chunks = search_top_chunks_bm25(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")
    # print(c["content"])


--- Rank 1 | score=8.59932413017356 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 58

--- Rank 2 | score=8.530439153128277 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 69

--- Rank 3 | score=8.011665099123933 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 16

--- Rank 4 | score=7.706528464312614 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 54

--- Rank 5 | score=7.622852374804522 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 45


In [5]:
question = "CATUElec"
chunks = search_top_chunks_bm25(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")
    # print(c["content"])


--- Rank 1 | score=5.894724379188959 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 91

--- Rank 2 | score=0.0 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-V2 | chunk: 0

--- Rank 3 | score=0.0 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-V2 | chunk: 1

--- Rank 4 | score=0.0 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-V2 | chunk: 2

--- Rank 5 | score=0.0 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-V2 | chunk: 3


In [6]:
question = "Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?"
chunks = search_top_chunks_bm25(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")



--- Rank 1 | score=12.903351595697474 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 31

--- Rank 2 | score=10.677321765598764 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 30

--- Rank 3 | score=8.726364420591603 ---
Doc: GDO-interventions-silos-VF-09-2019 | chunk: 26

--- Rank 4 | score=7.57910322755742 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 62

--- Rank 5 | score=7.100802960379735 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-V2 | chunk: 49


COMPARAISON RAG SEMANTIQUE VS RAG BM25

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from base_function import ask_question_with_semantic_rag, ask_question_with_BM25_rag, search_top_chunks_bm25, search_top_chunks_semantique

In [2]:
question = "Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?"

print("\n--- RAG Sémantique ---")
model_response, answer, memory = ask_question_with_semantic_rag(
    question,
    top_k=5
)
print(answer)

print("\n--- RAG BM25 ---")
model_response, answer, memory = ask_question_with_BM25_rag(
    question,
    top_k=5
)
print(answer)


--- RAG Sémantique ---
Pour envoyer les bons secours lors d'une intervention sur une éolienne, les informations indispensables sont les suivantes :

1. **Localisation** : Commune, nom du parc, numéro de l’éolienne, etc.
2. **Description de la problématique** : Type de sinistre, nombre de personnes en difficulté, leur pathologie et localisation.
3. **Nature du requérant** : Témoins, agent de maintenance, exploitant, centre de surveillance, etc.
4. **Facteurs aggravants** : Nombreux appels, conditions climatiques, heure, rassemblement à proximité, etc.
5. **Coordonnées du responsable de l’exploitation**.
6. **Modalités d’accès dans l’éolienne**.
7. **Caractéristiques des éoliennes installées** : Hauteur de mât, localisation des arrêts d’urgence, moyens de communication disponibles, etc.
8. **Chemins d’accès** : Pour faciliter l'arrivée des secours. 

Ces éléments permettent d'optimiser la prise d'appel et l'envoi des secours.

--- RAG BM25 ---
Je ne sais pas.


In [3]:
question = "Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?"

print("\n--- RAG Sémantique ---")
chunks = search_top_chunks_semantique(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")



print("\n--- RAG BM25 ---")
chunks = search_top_chunks_bm25(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")


--- RAG Sémantique ---

--- Rank 1 | score=0.73533726 ---
Doc: GDO_Interventions_dans_les_eoliennes_2019 | chunk: 13

--- Rank 2 | score=0.7123556 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 77

--- Rank 3 | score=0.69293684 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 76

--- Rank 4 | score=0.68889654 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 78

--- Rank 5 | score=0.6704755 ---
Doc: GDO_Interventions_dans_les_eoliennes_2019 | chunk: 15

--- RAG BM25 ---

--- Rank 1 | score=12.903351595697474 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 31

--- Rank 2 | score=10.677321765598764 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 30

--- Rank 3 | score=8.726364420591603 ---
Doc: GDO-interventions-silos-VF-09-2019 | chunk: 26

--- Rank 4 | score=7.57910322755742 ---
Doc: GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk: 62

--- Rank 5 | score=7.100802960379735 ---
Doc: GDO-Interventions-en-milieu-agricole-2019-